In [ ]:
# Parameters
kql_database_name = "kdb_appareliq"
interval_minutes = 15          # schedule cadence; the current snapshot is stamped at now() floored to this interval
backfill_hours = 24            # on each run, fill any missing hourly snapshots in this window (first run builds history)
lookback_hours = 24            # intraday window passed to fn_inventory_signal_snapshot
max_woc = 1.5                  # T1 thresholds (requirements 6.3)
min_intent = 1.3

# 06 - Intraday inventory signal snapshots (rt_inventory_signal) - SYNTHETIC

Appends snapshots of the intraday T1 logic (`fn_t1_intraday_at`) to `kdb_appareliq.rt_inventory_signal`: one row per
launch SKU-size x market per `signal_ts` with intraday available units, weeks of cover, intent index and the T1 flag.

* The table is bound to the ontology entity **InventoryPosition** as a time series, so the 2-source data agent
  (ontology + semantic model, decision D-18) can answer intraday questions without a direct KQL source.
* Idempotent: a snapshot is written only for slots that are not in the table yet. Runs on the Fabric job scheduler
  (every `interval_minutes`); the first run backfills hourly history for `backfill_hours`.
* Backfilled slots use the current Gold batch position plus the real-time events up to each slot.
* Runs as the caller's identity (no secrets). Pure Python kernel, standard library only.

In [ ]:
import datetime as dt, json, urllib.request
import notebookutils

ws_id = notebookutils.runtime.context["currentWorkspaceId"]


def fabric(path):
    req = urllib.request.Request(f"https://api.fabric.microsoft.com/v1{path}")
    req.add_header("Authorization", "Bearer " + notebookutils.credentials.getToken("pbi"))
    return json.loads(urllib.request.urlopen(req).read().decode())


db = next(d for d in fabric(f"/workspaces/{ws_id}/kqlDatabases")["value"] if d["displayName"] == kql_database_name)
uri = db["properties"]["queryServiceUri"]


def kql(csl, mgmt=False):
    req = urllib.request.Request(f"{uri}/v1/rest/{'mgmt' if mgmt else 'query'}", method="POST",
                                 data=json.dumps({"db": kql_database_name, "csl": csl}).encode())
    req.add_header("Authorization", "Bearer " + notebookutils.credentials.getToken("kusto"))
    req.add_header("Content-Type", "application/json")
    t = json.loads(urllib.request.urlopen(req).read().decode())["Tables"][0]
    names = [c["ColumnName"] for c in t["Columns"]]
    return [dict(zip(names, r)) for r in t["Rows"]]

In [ ]:
# Slots: the current interval plus hourly history; skip slots already written
now = dt.datetime.now(dt.timezone.utc).replace(second=0, microsecond=0, tzinfo=None)
current = now - dt.timedelta(minutes=now.minute % interval_minutes)
hour = now.replace(minute=0)
slots = sorted({current} | {hour - dt.timedelta(hours=h) for h in range(backfill_hours + 1)})
iso = lambda t: t.strftime("%Y-%m-%dT%H:%M:%SZ")
existing = {r["signal_ts"] for r in kql(f"rt_inventory_signal | where signal_ts >= datetime({iso(slots[0])}) | distinct signal_ts")}
todo = [s for s in slots if iso(s) not in existing]
print(f"{len(slots)} slots, {len(slots) - len(todo)} already present, writing {len(todo)}")
for s in todo:
    r = kql(f".set-or-append rt_inventory_signal <| fn_inventory_signal_snapshot(datetime({iso(s)}), "
            f"{lookback_hours}h, real({max_woc}), real({min_intent}))", mgmt=True)
    print(f"  {iso(s)}: {sum(x.get('RowCount', 0) for x in r)} rows")

In [ ]:
# Validate: one row per SKU-size x market per snapshot; T1 history for the demo story
check = kql("""rt_inventory_signal | where signal_ts > ago(2d)
| summarize rows = count(), keys = dcount(strcat(sku_id, '|', market_id)), t1 = countif(t1_intraday) by signal_ts
| summarize snapshots = count(), duplicate_snapshots = countif(rows != keys), max_rows = max(rows),
            snapshots_with_t1 = countif(t1 > 0), latest = max(signal_ts)""")[0]
print(check)
assert check["snapshots"] > 0 and check["duplicate_snapshots"] == 0, "rt_inventory_signal snapshot keys are not unique"
for r in kql("""rt_inventory_signal | where signal_ts > ago(2d) and t1_intraday
| summarize sku_sizes = dcount(sku_id), first_seen = min(signal_ts), last_seen = max(signal_ts) by style_id, market_id
| top 5 by sku_sizes"""):
    print(r)